# 03 — Thermal envelope and baseline epoch

Two quantities the rest of the analysis rests on:

1. Each district's thermal envelope (mu, sigma), fit on the lapse-rate surface that suitability is evaluated against — not on a different temperature product, which was a defect in an earlier version.
2. **When "present climate" actually is.** The temperature field is a 1990–2021 climatology with midpoint ~2006, so its zero point is not today. Peak offsets measured against it need re-centring on the end of the observational record before they can be read as statements about now.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, glob, re, rasterio, json
import pipeline as P

isl = P.load_island()
FT, FTemp, reg, xy = P.load_farms(isl)
print(f"island land cells {isl['X'].shape[0]:,}   farm cells {len(FT)} "
      f"(kona {(reg=='kona').sum()}, kau {(reg=='kau').sum()})")

island land cells 42,524   farm cells 471 (kona 409, kau 62)


## Thermal envelopes

Fit on the farm cells' own temperatures. `sigma` carries a 1.5x inflation, matching the published definition; the `tau = 0.5` level set then spans `mu ± sqrt(2 ln 2) * sigma`.

In [2]:
idx = np.arange(len(FT))
rows = []
for r in ('kona', 'kau'):
    mu, sg = P.envelope(FTemp, reg, idx, r)
    rows.append([r, mu, sg, mu - P.HW*sg, mu + P.HW*sg])
t = pd.DataFrame(rows, columns=['district','mu (C)','sigma (C)','band lo','band hi'])
print(t.round(3).to_string(index=False))
print(f"\nthermal optima differ by {abs(rows[0][1]-rows[1][1]):.2f} C -- "
      "the districts occupy effectively the same thermal niche.")

district    mu (C)  sigma (C)  band lo  band hi
    kona 20.999001      1.371   19.385   22.613
     kau 20.823000      1.040   19.598   22.048

thermal optima differ by 0.18 C -- the districts occupy effectively the same thermal niche.


## Baseline epoch — how far the record has already moved

Sampling all 384 monthly rasters at the farm cells gives a per-year belt-mean temperature. Regressing on year measures the warming already present *inside the product's own baseline window*, which is what the climatological mean averages away.

In [3]:
import geopandas as gpd
allf = pd.read_pickle('../data/plot_all_features.pkl')
fp = gpd.GeoDataFrame(allf[allf.label==1].copy(), geometry='geometry', crs='EPSG:4326')
cen = fp.geometry.centroid
pts = list(zip(cen.x.values, cen.y.values))

files = sorted(f for f in glob.glob('../data/kodama_temperature/*.tif')
               if re.search(r'_\d{4}-\d{2}\.tif$', f))
yr_sum, yr_n = {}, {}
for f in files:
    y = int(re.search(r'_(\d{4})-(\d{2})\.tif$', f).group(1))
    with rasterio.open(f) as s:
        v = np.array([x[0] for x in s.sample(pts)], dtype=float)
        v[v == s.nodata] = np.nan
    yr_sum[y] = yr_sum.get(y, 0.0) + np.nanmean(v); yr_n[y] = yr_n.get(y, 0) + 1
years = np.array(sorted(k for k in yr_sum if yr_n[k] == 12))
belt  = np.array([yr_sum[y]/12 for y in years])
print(f'{len(years)} complete years, {years.min()}-{years.max()}')

32 complete years, 1990-2021


In [4]:
A = np.vstack([years - years.mean(), np.ones_like(years)]).T
coef, *_ = np.linalg.lstsq(A, belt, rcond=None)
resid = belt - A @ coef
se = np.sqrt((resid**2).sum() / (len(years)-2) / ((years-years.mean())**2).sum())
trend, trend_se = coef[0]*10, se*10
mid = years.mean()
offset = trend/10 * (years.max() - mid)
offset_se = trend_se/10 * (years.max() - mid)

print(f'belt trend      {trend:+.3f} +/- {trend_se:.3f} C/decade')
print(f'climatological midpoint {mid:.0f}, end of record {years.max()}')
print(f'offset mean -> end of record  {offset:+.3f} +/- {offset_se:.3f} C')
print(f'\nfor scale, notebook 02 measured the annual temperature cycle at 3.23 C,')
print(f'so the record has already moved {100*offset/3.23:.0f}% of one annual swing.')

json.dump({'belt_trend_per_decade': float(trend), 'belt_trend_se': float(trend_se),
           'offset_to_endrec': float(offset), 'offset_to_endrec_se': float(offset_se),
           'clim_midpoint': float(mid)}, open('data/baseline_epoch.json','w'), indent=1)
print('\nwrote data/baseline_epoch.json')

belt trend      +0.313 +/- 0.065 C/decade
climatological midpoint 2006, end of record 2021
offset mean -> end of record  +0.486 +/- 0.100 C

for scale, notebook 02 measured the annual temperature cycle at 3.23 C,
so the record has already moved 15% of one annual swing.

wrote data/baseline_epoch.json


## Figure — "present climate" is not the present

The climatological mean averages over a period that was itself warming. Measured against the end of the observational record, the belt has already moved most of half a degree.

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
os.makedirs('figures', exist_ok=True)
KONA, KAU, F_COL, C_COL = '#c1440e', '#1f6f8b', '#c1440e', '#888888'
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})

fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))
ax = axes[0]
ax.plot(years, belt, 'o-', color='#555555', ms=3, lw=1)
ax.plot(years, A @ coef, color=F_COL, lw=2,
        label=f'{trend:+.3f} ± {trend_se:.3f} °C/decade')
ax.axhline(belt.mean(), color='k', ls=':', lw=1, label='climatological mean')
ax.set_xlabel('year'); ax.set_ylabel('belt mean temperature (°C)')
ax.set_title('Warming inside the baseline window'); ax.legend(frameon=False, fontsize=9)

ax = axes[1]
tt = np.linspace(16, 26, 400)
for r, col in (('kona', KONA), ('kau', KAU)):
    mu, sg = P.envelope(FTemp, reg, idx, r)
    ax.plot(tt, np.exp(-0.5*((tt-mu)/sg)**2), color=col, lw=2, label=f'{r} (μ={mu:.2f})')
    ax.axvspan(mu-P.HW*sg, mu+P.HW*sg, color=col, alpha=.07)
ax.axhline(0.5, color='k', ls=':', lw=1)
ax.set_xlabel('temperature (°C)'); ax.set_ylabel('suitability')
ax.set_title('Thermal envelopes — effectively the same niche')
ax.legend(frameon=False, fontsize=9)
fig.tight_layout(); fig.savefig('figures/03_baseline_envelope.png', dpi=200, bbox_inches='tight')
plt.close(fig); print('figures/03_baseline_envelope.png')

figures/03_baseline_envelope.png
